# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime, timedelta
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen_Downloads/"
temp_files = downloads + "Andersen_Temp_Files/"
complete_files = downloads + "Andersen_Complete_Files/"

os.chdir(downloads)

In [ ]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# print(len(metadata)) # 7397 rows

# Split date format to only check year
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: x.split("-")[0])
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: int(x))

# Find only >= 2024 using run ID from metadata
metadata_new = metadata[metadata["Collection_Date"] >= 2024]
metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats

# Find only >= last date using Release Date from metadata 
metadata_new["ReleaseDate"] = metadata_new["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata_new = metadata_new[metadata_new["ReleaseDate"] >= datetime(2025, 4, 15).strftime("%Y-%m-%d")]

print(len(metadata_new)) # 6053 rows between 1/1/2024 and 4/14/2025

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_19100\80415036.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats


7044


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_19100\80415036.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_new["ReleaseDate"] = metadata_new["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))


### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype: B3.13 or D1.1]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

In [20]:
# # Get genotype from genoflu
# os.chdir(temp_files)
# output_tsv = pd.read_csv("output.tsv", delimiter="\t")

# b313_and_d11_only = output_tsv[(output_tsv["Genotype"] == "B3.13") | (output_tsv["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")
# # print(b313_and_d11_only)
# print(len(b313_and_d11_only)) # 5160 rows

# metadata_new = metadata_new.merge(b313_and_d11_only, on="Run", how="inner")

# print(len(metadata_new)) # 5160

In [21]:
# Get genotype from genoflu_results.tsv

output_tsv = pd.read_csv("genoflu_results.tsv", delimiter="\t")

# b313_and_d11_only = output_tsv[(output_tsv["Genotype"] == "B3.13") | (output_tsv["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")
# # print(b313_and_d11_only)
# print(len(b313_and_d11_only)) 

# metadata_new = metadata_new.merge(b313_and_d11_only, on="Run", how="inner")

metadata_new["Genotype"] = output_tsv["Genotype"]
metadata_new = metadata_new[~metadata_new["Genotype"].str.contains('Not assigned')]

print(len(metadata_new)) 

6699


In [22]:
print(len(metadata_new))

print(metadata_new)

6699
              Run Assay Type  AvgSpotLen      Bases    BioProject  \
1     SRR28752447        WGS      241.29   86080323  PRJNA1102327   
2     SRR28752448        WGS      250.30   75035343  PRJNA1102327   
3     SRR28752449        WGS      146.61   59363690  PRJNA1102327   
4     SRR28752450        WGS      251.31  119232569  PRJNA1102327   
7     SRR28752453        WGS      144.71   37055919  PRJNA1102327   
...           ...        ...         ...        ...           ...   
8383  SRR33029748        WGS      147.39  252799915   PRJNA980729   
8384  SRR33029749        WGS      147.88  167615693   PRJNA980729   
8385  SRR33029750        WGS      146.73   95329644   PRJNA980729   
8386  SRR33029751        WGS      146.12  269328456   PRJNA980729   
8387  SRR33029752        WGS      145.33  270721593   PRJNA980729   

         BioSample BioSampleModel     Bytes Center Name  Collection_Date  ...  \
1     SAMN41019237          Viral  28164109   USDA-NVSL             2024  ...   
2   

In [23]:
# Get geolocation from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first")
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2])

metadata_genbank = metadata_new.merge(genbank_mapping, on=["Run"], how="inner")
# metadata_new["name_state"] = "unknown"
# metadata_genbank = metadata_new

print(genbank_mapping["name_state"])
print(len(metadata_genbank))
display(metadata_genbank)

0        Texas
8        Texas
16       Texas
24       Texas
32       Texas
         ...  
37270       OH
37278       OH
37286       OH
37294       OH
37302       MT
Name: name_state, Length: 4103, dtype: object
3896


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,retraction_detection_date_utc,Genotype,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state
0,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,NaN,D1.3,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,SRR28752447,HA,PP752829.1,4,A/cattle/Texas/24-009108-005/2024,Texas
1,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,NaN,B3.13,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4,A/cattle/Texas/24-009108-004/2024,Texas
2,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,NaN,B3.13,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4,A/cattle/Texas/24-009108-003/2024,Texas
3,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,NaN,B3.13,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4,A/cattle/Texas/24-009108-002/2024,Texas
4,SRR28752453,WGS,144.71,37055919,PRJNA1102327,SAMN41019231,Viral,12938409,USDA-NVSL,2024,...,NaN,B3.13,SRR28752453_HA_cns.fa,Consensus_SRR28752453_HA_cns_threshold_0.5_qua...,SRR28752453,HA,PP752677.1,4,A/cattle/Texas/24-009088-001/2024,Texas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3891,SRR32633088,WGS,145.11,67016467,PRJNA1102327,SAMN47290851,Viral,25388495,USDA-NVSL,2025,...,NaN,B3.13,SRR32633088_HA_cns.fa,Consensus_SRR32633088_HA_cns_threshold_0.5_qua...,SRR32633088,HA,PV456280.1,4,A/cat/OR/25-005913-003-original/2025,OR
3892,SRR32633089,WGS,148.14,140023511,PRJNA1102327,SAMN47290850,Viral,52157519,USDA-NVSL,2025,...,NaN,B3.13,SRR32633089_HA_cns.fa,Consensus_SRR32633089_HA_cns_threshold_0.5_qua...,SRR32633089,HA,PV456272.1,4,A/cat/OR/25-005800-002-original/2025,OR
3893,SRR32633090,WGS,148.49,83727737,PRJNA1102327,SAMN47290849,Viral,31619233,USDA-NVSL,2025,...,NaN,D1.3,SRR32633090_HA_cns.fa,Consensus_SRR32633090_HA_cns_threshold_0.5_qua...,SRR32633090,HA,PV456264.1,4,A/cat/OR/25-005800-001-original/2025,OR
3894,SRR32633093,WGS,148.10,105910536,PRJNA1102327,SAMN47290846,Viral,39757708,USDA-NVSL,2025,...,NaN,B3.2,SRR32633093_HA_cns.fa,Consensus_SRR32633093_HA_cns_threshold_0.5_qua...,SRR32633093,HA,PV457240.1,4,A/cattle/CA/25-005677-001-original/2025,CA


In [24]:

metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank))
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["years"]

SAMN41019237
SAMN41019236
SAMN41019235
SAMN41019234
SAMN41019231
SAMN41019230
SAMN41019229
SAMN41019228
SAMN41019183
SAMN41019227
SAMN41019226
SAMN41019225
SAMN41019224
SAMN41019222
SAMN41019221
SAMN41019220
SAMN41019219
SAMN41019218
SAMN41019182
SAMN41019217
SAMN41019215
SAMN41019213
SAMN41019212
SAMN41019209
SAMN41019366
SAMN41019365
SAMN41019364
SAMN41019363
SAMN41019362
SAMN41019361
SAMN41019359
SAMN41019358
SAMN41019196
SAMN41019357
SAMN41019355
SAMN41019354
SAMN41019353
SAMN41019352
SAMN41019351
SAMN41019350
SAMN41019349
SAMN41019348
SAMN41019195
SAMN41019347
SAMN41019346
SAMN41019345
SAMN41019344
SAMN41019343
SAMN41019342
SAMN41019341
SAMN41019340
SAMN41019339
SAMN41019338
SAMN41019277
SAMN41019274
SAMN41019273
SAMN41019272
SAMN41019271
SAMN41019270
SAMN41019269
SAMN41019268
SAMN41019187
SAMN41019208
SAMN41019181
Unable to find collection date.
SAMN41019207
SAMN41019206
SAMN41019205
SAMN41019204
SAMN41019202
SAMN41019416
SAMN41019415
SAMN41019414
SAMN41019413
SAMN41019412
SAMN41

In [ ]:
# # Upload saved data
# os.chdir(temp_files)
# metadata_genbank = pd.read_csv("metadata_genbank.csv")

In [25]:
# Save this so we don't have to do it again

os.chdir(temp_files)
metadata_genbank.to_csv("metadata_genbank.csv")

In [26]:

unique_animals_all = sort_animals_andersen(metadata_genbank)

# Flatten unique_animals
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal))
# animals_df = pd.DataFrame(columns=["avian", "cattle", "feline", "other_mammal", "human", "other"])
# animals_df["other"] = unique_animals_set # to sort

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv")


# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv")


['falcon', 'cago', 'red fox', 'cackling goose', 'wood duck', 'emu', 'owl', 'american wigeon', 'pheasant', 'alpaca', 'peafowl', 'burrowing owl', 'american robin', 'merganser', 'common raven', 'raw pet food', 'cattle milk product', 'quail', 'dove', 'duck', 'chicken', 'bald eagle', 'domestic cat', 'bottlenose dolphin', 'guineafowl', 'barn owl', 'hosp', 'fox', 'guinea fowl', 'red tailed hawk', 'serval', 'skunk', 'white-winged dove', 'swan', "geoffroy's cat", 'hooded merganser', 'american woodcock', 'pig', 'pet food', 'house mouse', 'blackbird', 'pefa', 'western sandpiper', 'raccoon', 'trumpeter swan', 'house sparrow', 'goat', 'black-crowned night-heron', 'ganada goose', 'pelican', 'gadwall', 'lynx', 'vulture', 'great horned owl', 'mountain lion', 'cattle', 'crane', 'comon-grackle', 'ermine', 'great blue heron', "cooper's hawk", 'red-breasted merganser', 'lion', 'american crow', 'snowy owl', 'canada goose', 'red-tailed hawk', 'house-mouse', 'avian', 'rock pigeon', 'pigeon', 'mute swan', 'tu

In [27]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_genbank, animals_ref) # Get host type
metadata_genbank["years"] = metadata_genbank["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

In [ ]:
for num, collection_date in enumerate(metadata_genbank["Collection_Date_Specific"]):
    if collection_date != collection_date: # If nan
        metadata_genbank.loc[num, "Collection_Date_Specific"] = metadata_genbank.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata_genbank.loc[num, "Collection_Date_Specific"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata_genbank.loc[num, "Collection_Date_Specific"] = date

# Make names

names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Collection_Date_Specific"].apply(lambda x: str(x)) + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

# print(metadata_genbank["years"])

metadata_genbank.to_csv("metadata_genbank_named.csv")

# display(metadata_genbank)

0       2024
1       2024
2       2024
3       2024
4       2024
        ... 
3891    2025
3892    2025
3893    2025
3894    2025
3895    2025
Name: years, Length: 3896, dtype: object


In [30]:
# Get list of genotypes

os.chdir(home)

genotypes_df = pd.read_excel("genotype_key.xlsx")

genotypes = list(genotypes_df["Genotype"])

print(genotypes)

['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'B1.1', 'B1.2', 'B1.3', 'B2.1', 'B2.2', 'B3.1', 'B3.2', 'B3.3', 'B3.4', 'B3.5', 'B3.6', 'B4.1', 'B5.1', 'Minor01', 'Minor04', 'Minor07', 'Minor08', 'Minor09', 'Minor10', 'Minor11', 'Minor12', 'Minor13', 'Minor14', 'Minor15', 'Minor16', 'Minor17', 'Minor18', 'Minor19', 'Minor24', 'Minor25', 'Minor26', 'Minor27', 'Minor28', 'Minor29', 'Minor30', 'Minor31', 'Minor32', 'Minor33', 'Minor34', 'Minor35', 'Minor36', 'Minor37', 'Minor38', 'Minor39', 'Minor40', 'Minor41', 'Minor42', 'Minor43', 'Minor44', 'Minor45', 'Minor46', 'Minor47', 'Minor48', 'B3.7', 'Minor50', 'Minor51', 'C1.1', 'Minor52', 'Minor53', 'B3.11', 'Minor55', 'Minor56', 'Minor57', 'Minor58', 'B3.10', 'C2.1', 'Minor60', 'Minor61', 'B3.8', 'Minor62', 'Minor63', 'B3.12', 'Minor65', 'Minor66', 'Minor67', 'B3.9', 'Minor70', 'Minor71', 'B3.13', 'Minor73', 'Minor74', 'Minor75', 'Minor76', 'Minor77', 'Minor78', 'Minor79', 'Minor80', 'Minor81', 'Minor82', 'Minor83', 'Minor84', 'C3.1', 'Minor86', 'Mino

In [31]:
# Make fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

pairs = []
fasta_files = {}

for genotype in genotypes:
    for segment in ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()

In [32]:
# Create fasta files 

date = "2025-04-14"

os.chdir(temp_files)

for pair in fasta_files.keys():
    output_path = temp_files + pair + "_andersen_" + date + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

>A/Cattle/Texas/24-009028-019-original/2024|H5N1|2024-03-20|cattle|A1
>A/Cattle/Texas/24-009775-001-original/2024|H5N1|2024-03-10|cattle|A1
>A/Cattle/New Mexico/24-010512-001-original/2024|H5N1|2024-04-02|cattle|A1
>A/Cattle/South Dakota/24-010354-020-original/2024|H5N1|2024-04-05|cattle|A1
>A/Cattle/Texas/24-009110-019-original/2024|H5N1|2024-03-14|cattle|A1
>A/Chicken/Texas/24-008355-002-original/2024|H5N1|2024-03-16|avian|A1
>A/Cattle/South Dakota/24-010354-001-original/2024|H5N1|2024-04-05|cattle|A1
>A/Cattle/Texas/24-009367-005-original/2024|H5N1|2024-03-25|cattle|A1
>A/CATTLE/New Mexico/24-010192-003/2024|H5N1|2024-04-01|cattle|A1
>A/American Crow/Illinois/24-004479-002-original/2024|H5N1|2024-02-08|avian|A1
>A/DOMESTIC-CAT/NM/24-012046-003/2024|H5N1|2024-04-17|feline|A1
>A/TURKEY/MI/24-012707-001/2024|H5N1|2024-04-29|avian|A1
>A/CATTLE/Idaho/24-012341-022-original/2024|H5N1|2024-04-23|cattle|A1
>A/CATTLE/Colorado/24-012225-014/2024|H5N1|2024-04-23|cattle|A1
>A/CHICKEN/Minnesota/

In [39]:
# De-duplication 

# Gisaid 

gisaid = downloads + "GISAID_Complete_Fasta_Files/01-01-2024--03-31-2025_renamed/"

os.chdir(gisaid)



In [46]:
dfs_gisaid = create_dataframes(gisaid)
print(dfs_gisaid["B3.13_HA"][0])

IndexError: list index out of range

In [41]:
# Do the same with Andersen 

dfs_andersen = create_dataframes(temp_files)

In [42]:
# print(dfs_andersen["D1.3_HA"][0])

In [43]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    for i, df in enumerate(dataframes):
        print(i)
        try:
            full_df = df.merge(dfs_gisaid[key][i], how="outer")
            # print(full_df)
            full_df = full_df.drop_duplicates(subset=["isolate_partial"])
            full_dfs[key].append(full_df)
        except:
            print("Failed to merge dataframes in ", key)



0
Failed to merge dataframes in  A1_HA_andersen_2025-04-14
0
Failed to merge dataframes in  A1_MP_andersen_2025-04-14
0
Failed to merge dataframes in  A1_NA_andersen_2025-04-14
0
Failed to merge dataframes in  A1_NP_andersen_2025-04-14
0
Failed to merge dataframes in  A1_NS_andersen_2025-04-14
0
Failed to merge dataframes in  A1_PA_andersen_2025-04-14
0
Failed to merge dataframes in  A1_PB1_andersen_2025-04-14
0
Failed to merge dataframes in  A1_PB2_andersen_2025-04-14
0
Failed to merge dataframes in  A2_HA_andersen_2025-04-14
0
Failed to merge dataframes in  A2_MP_andersen_2025-04-14
0
Failed to merge dataframes in  A2_NA_andersen_2025-04-14
0
Failed to merge dataframes in  A2_NP_andersen_2025-04-14
0
Failed to merge dataframes in  A2_NS_andersen_2025-04-14
0
Failed to merge dataframes in  A2_PA_andersen_2025-04-14
0
Failed to merge dataframes in  A2_PB1_andersen_2025-04-14
0
Failed to merge dataframes in  A2_PB2_andersen_2025-04-14
0
Failed to merge dataframes in  A3_HA_andersen_2025

In [44]:
# If none in one database, only use the other and drop duplicates

full_dfs = defaultdict(list)
for key in dfs_andersen.keys():
    print(key)
# for key in ["D1.3"]:
    dataframes = dfs_andersen[key]
    for i, df in enumerate(dataframes):
        print(i)
        try:
            # full_df = df.merge(dfs_gisaid[key][i], how="outer")
            # print(full_df)
            full_df = full_df.drop_duplicates(subset=["isolate_partial"])
            full_dfs[key].append(full_df)
        except:
            print("Failed to merge dataframes in ", key)

A1_HA_andersen_2025-04-14
0
Failed to merge dataframes in  A1_HA_andersen_2025-04-14
A1_MP_andersen_2025-04-14
0
Failed to merge dataframes in  A1_MP_andersen_2025-04-14
A1_NA_andersen_2025-04-14
0
Failed to merge dataframes in  A1_NA_andersen_2025-04-14
A1_NP_andersen_2025-04-14
0
Failed to merge dataframes in  A1_NP_andersen_2025-04-14
A1_NS_andersen_2025-04-14
0
Failed to merge dataframes in  A1_NS_andersen_2025-04-14
A1_PA_andersen_2025-04-14
0
Failed to merge dataframes in  A1_PA_andersen_2025-04-14
A1_PB1_andersen_2025-04-14
0
Failed to merge dataframes in  A1_PB1_andersen_2025-04-14
A1_PB2_andersen_2025-04-14
0
Failed to merge dataframes in  A1_PB2_andersen_2025-04-14
A2_HA_andersen_2025-04-14
0
Failed to merge dataframes in  A2_HA_andersen_2025-04-14
A2_MP_andersen_2025-04-14
0
Failed to merge dataframes in  A2_MP_andersen_2025-04-14
A2_NA_andersen_2025-04-14
0
Failed to merge dataframes in  A2_NA_andersen_2025-04-14
A2_NP_andersen_2025-04-14
0
Failed to merge dataframes in  A2

In [47]:
print(dfs_andersen["D1.3_HA"]) #[0])

[]


In [ ]:
# Create fasta files 

os.chdir(complete_files)
for pair in full_dfs.keys():
    output_path = complete_files + pair + "_combined_" + date + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
        # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

In [52]:
os.chdir(complete_files)

end_date = "2025-04-14"

segment_files = []
for dirpath, dirs, files in os.walk(temp_files): # + "01-01-2024--03-31-2025_renamed/"):
    for segment in ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]:
        base = []
        for file in files:
            file_name = os.path.join(dirpath, file)
            # print(file_name)
            if segment in file_name:
                with open(file_name) as f:
                    lines = f.readlines()
                    for line in lines:
                        base.append(line)
                f.close()
        os.chdir(complete_files)
        output_path = complete_files + "All_Genotypes_" + segment + "_andersen_" + end_date + ".fasta" # Genotype and Segment should all be the same
        output_file = open(output_path, "w")
        for b in base:
            output_file.write(b)
        output_file.close()

In [ ]:
# Get new collection dates for Alvin's file

os.chdir(downloads + "Combined_Files/")

headers = []
for dirpath, dirs, files in os.walk(downloads + "Combined_Files/"):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if "newid-B3.13_genome_APR14_trim_aln_w_incomplete_dates_n1053" in file_name:
            with open(file_name) as f:
                lines = f.readlines()
                for num, line in enumerate(lines):
                    if line[0] == ">":
                        header = line
                        headers.append(header)

print(headers[0])

biosamples = []
isolate_partial = []
for header in headers:
    # print(header)
    id = header.split("/")[3]
    partial = id.split("_")[-1] # If 25_, get the last bit
    digits = partial.split("-")
    isolate = ""
    other = ""
    for d in digits:
        # print(d)
        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
            isolate = d + "-"
        elif len(d) == 3 and d.isnumeric():
            isolate = isolate + d
        elif d.isnumeric() == False: # If it's a weird isolate
            other = d + "-"
        else: 
            other = other + d
    # Now add to list to check in Andersen files without doing wild for loops
    if len(isolate) == 10: # If this is a correctly formatted isolate
        # isolates.append(isolate)
        # All headers are followed by sequences
        isolate_partial.append(isolate)
    else: # If this is some other isolate
        isolate_partial.append(other)
    # print(id)

print(isolate_partial)

>A/ALPACA/Idaho/24-014328-007/2024|H5N1|2024|other_mammal|B3.13

['014328-007', '008304-003', '014336-001', '014336-002', '014336-003', '014336-004', '014336-006', '014336-007', '014336-008', '014336-010', '014336-011', '014336-012', '014336-013', '014336-014', '014767-003', '014769-001', '014769-002', '014769-003', '013787-001', '013787-002', '013789-001', '013789-002', '013789-003', '013789-004', '013789-005', '013789-006', '013789-007', '013789-009', '013789-010', '013789-011', '013789-012', '013999-001', '013999-003', '013999-005', '013999-006', '013999-007', '014001-001', '014001-002', '014001-004', '014287-001', '014510-003', '014510-007', '033204-002', '036210-004', '036421-002', '001948-005', '003670-004', '004631-003', '004631-004', '006506-001', '006506-003', '006506-005', '006507-001', '006507-002', '006507-003', '006507-004', '006507-005', '006508-001', '006508-002', '006508-003', '006508-005', '006508-006', '006508-007', '006508-008', '006508-009', '006508-010', '006508-01